# Phase 2 — Data Cleaning & Filtering (rules + transformers)

Turns the Phase-1 findings into **reusable code**, never manual edits:

* **Stateless rules** (learn nothing from the data distribution) → one function `apply_stateless_rules(df)`.
  Safe to run before the split. Each rule logs how many rows it affects.
* **Learned cleaning** → sklearn-compatible transformers / helpers that are **defined here but fit later,
  on X_train only** (Phase 5). Nothing in this notebook is fitted on real data for use downstream.
* **No data is saved** and **no split is made here** — that happens exactly once in Phase 5.
  The two cells marked `COPY VERBATIM INTO PHASE 5` are the single source of truth for cleaning.

Colab: upload `House_Rent_Dataset.csv` into the working directory, then *Runtime → Run all*.

In [1]:
import warnings
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 140)
RANDOM_STATE = 42
TARGET = "Rent"

raw = pd.read_csv("House_Rent_Dataset.csv")
print("Raw shape:", raw.shape)

Raw shape: (4746, 12)


## 1. Stateless rules — `COPY VERBATIM INTO PHASE 5`

| Rule | What | Why (Phase-1 ref) | Distribution-free? |
|---|---|---|---|
| R1 `strip_text` | trim whitespace on all text columns; `Area Locality` → lower-case, punctuation→space, collapse spaces | D10: 4 localities with stray whitespace, case/punct variants | yes — pure string op |
| R2 `validity` | keep rows with `Rent>0`, `Size>0`, `BHK>=1`, `Bathroom>=1` | definitional sanity (guard; 0 rows today) | yes — definitional bounds, not read off a chart |
| R3 `dedup_relisted` | `drop_duplicates` on **all columns except `Posted On`** (keep first) | D3: same listing re-posted → could sit in train *and* test | yes |
| R4 `parse_floor` | `Floor` → `current_floor` (Ground=0, Upper Basement=−1, Lower Basement=−2), `total_floors` (NaN if absent); if current>total, total := current | D4: 4 rows lack "out of", 2 rows have current>total (a building has at least as many floors as the unit's floor) | yes |
| R5 `parse_date` | `Posted On` → datetime `posted_date` | D5: needed for date features in Phase 4 | yes |

Raw `Floor` / `Posted On` strings are **kept** here; whether they are dropped is finalised in Phase 4 when
their replacement features are chosen.

In [2]:
# ======================= PHASE-2 STATELESS RULES — COPY VERBATIM INTO PHASE 5 =======================
TEXT_COLS = ["Posted On", "Floor", "Area Type", "Area Locality", "City",
             "Furnishing Status", "Tenant Preferred", "Point of Contact"]
FLOOR_LEVEL_MAP = {"Ground": 0, "Upper Basement": -1, "Lower Basement": -2}


def rule_strip_text(df):
    df = df.copy()
    for c in TEXT_COLS:
        df[c] = df[c].astype(str).str.strip()
    df["Area Locality"] = (df["Area Locality"].str.lower()
                           .str.replace(r"[^a-z0-9]+", " ", regex=True)
                           .str.strip())
    return df


def rule_validity(df):
    ok = (df["Rent"] > 0) & (df["Size"] > 0) & (df["BHK"] >= 1) & (df["Bathroom"] >= 1)
    return df[ok]


def rule_dedup_relisted(df):
    return df.drop_duplicates(subset=[c for c in df.columns if c != "Posted On"], keep="first")


def rule_parse_floor(df):
    df = df.copy()
    parts = df["Floor"].str.split(" out of ", n=1, expand=True).reindex(columns=[0, 1])
    cur = parts[0].replace(FLOOR_LEVEL_MAP)
    df["current_floor"] = pd.to_numeric(cur, errors="coerce").astype("float64")
    df["total_floors"] = pd.to_numeric(parts[1], errors="coerce").astype("float64")
    bad = df["current_floor"] > df["total_floors"]          # NaN comparisons are False
    df.loc[bad, "total_floors"] = df.loc[bad, "current_floor"]
    return df


def rule_parse_date(df):
    df = df.copy()
    df["posted_date"] = pd.to_datetime(df["Posted On"], format="%Y-%m-%d", errors="coerce")
    return df


STATELESS_RULES = [rule_strip_text, rule_validity, rule_dedup_relisted, rule_parse_floor, rule_parse_date]


def apply_stateless_rules(df, verbose=True):
    """Apply all distribution-free rules in a fixed order. Returns (clean_df, log_df)."""
    log = []
    for rule in STATELESS_RULES:
        n_before = len(df)
        df = rule(df)
        log.append({"rule": rule.__name__, "rows_before": n_before, "rows_after": len(df),
                    "rows_removed": n_before - len(df)})
    log = pd.DataFrame(log)
    if verbose:
        print(log.to_string(index=False))
    return df.reset_index(drop=True), log
# =====================================================================================================

## 2. Apply the rules (in memory only) and audit their effect

In [3]:
clean, rule_log = apply_stateless_rules(raw)
print(f"\nRows: {len(raw)} -> {len(clean)}  (removed {len(raw) - len(clean)})")
assert raw.shape == (4746, 12), "raw frame must be untouched"

               rule  rows_before  rows_after  rows_removed
    rule_strip_text         4746        4746             0
      rule_validity         4746        4746             0
rule_dedup_relisted         4746        4737             9
   rule_parse_floor         4737        4737             0
    rule_parse_date         4737        4737             0

Rows: 4746 -> 4737  (removed 9)


In [4]:
# R1 — what changed in text?
r1 = rule_strip_text(raw)
loc_changed = int((r1["Area Locality"] != raw["Area Locality"]).sum())
print("R1: localities whose text changed (incl. lower-casing):", loc_changed,
      "| unique localities:", raw["Area Locality"].nunique(), "->", r1["Area Locality"].nunique())
print("R1: other text columns changed:",
      {c: int((r1[c] != raw[c]).sum()) for c in TEXT_COLS if c != "Area Locality"})

# R3 — which rows were treated as re-listings? (shown, removed by rule — not by index)
key = [c for c in r1.columns if c != "Posted On"]
dups = r1[r1.duplicated(subset=key, keep=False)].sort_values(key)
print(f"\nR3: {r1.duplicated(subset=key).sum()} re-listed rows removed; groups shown below")
display(dups[["Posted On", "BHK", "Rent", "Size", "Floor", "Area Locality", "City"]])

R1: localities whose text changed (incl. lower-casing): 4695 | unique localities: 2235 -> 2228
R1: other text columns changed: {'Posted On': 0, 'Floor': 0, 'Area Type': 0, 'City': 0, 'Furnishing Status': 0, 'Tenant Preferred': 0, 'Point of Contact': 0}

R3: 9 re-listed rows removed; groups shown below


,Posted On,BHK,Rent,Size,Floor,Area Locality,City
133,2022-07-01,1,4700,250,1 out of 2,kaikhali,Kolkata
203,2022-06-12,1,4700,250,1 out of 2,kaikhali,Kolkata
2580,2022-05-23,1,8000,200,2 out of 4,kst chattarpur apartments,Delhi
2581,2022-05-27,1,8000,200,2 out of 4,kst chattarpur apartments,Delhi
2302,2022-06-29,1,13000,700,1 out of 5,gh society,Bangalore
2380,2022-06-29,1,13000,700,1 out of 5,gh society,Bangalore
2535,2022-06-04,1,20000,400,3 out of 4,lajpat nagar 4,Delhi
2793,2022-05-27,1,20000,400,3 out of 4,lajpat nagar 4,Delhi
2758,2022-06-04,2,13000,400,2 out of 4,kst chattarpur apartments,Delhi
2837,2022-05-14,2,13000,400,2 out of 4,kst chattarpur apartments,Delhi


In [5]:
# R4 — parse audit
p = rule_parse_floor(rule_strip_text(raw))
print("current_floor NaN:", int(p["current_floor"].isna().sum()), "| total_floors NaN:", int(p["total_floors"].isna().sum()))
print("current_floor range:", p["current_floor"].min(), "to", p["current_floor"].max(),
      "| total_floors range:", p["total_floors"].min(), "to", p["total_floors"].max())
odd = ~raw["Floor"].str.contains(" out of ", regex=False) | (p["current_floor"] > pd.to_numeric(
      raw["Floor"].str.split(" out of ", n=1, expand=True)[1], errors="coerce"))
print("Rows hit by the 'no total' / 'current > total' sub-rules:")
display(p.loc[odd, ["Floor", "current_floor", "total_floors", "BHK", "Rent", "City"]])
print("Any current_floor > total_floors left:", int((p["current_floor"] > p["total_floors"]).sum()))

# R5 — date audit
print("posted_date NaT:", int(clean["posted_date"].isna().sum()),
      "| range:", clean["posted_date"].min().date(), "->", clean["posted_date"].max().date())
print("\nMissing values after rules:\n", clean.isna().sum()[lambda s: s > 0].to_string())

current_floor NaN: 0 | total_floors NaN: 4
current_floor range: -2.0 to 76.0 | total_floors range: 1.0 to 89.0
Rows hit by the 'no total' / 'current > total' sub-rules:


,Floor,current_floor,total_floors,BHK,Rent,City
105,8 out of 5,8.0,8.0,1,6000,Kolkata
161,2 out of 1,2.0,2.0,2,10000,Kolkata
2553,3,3.0,NaN,2,20000,Delhi
2883,Ground,0.0,NaN,1,18000,Delhi
4490,1,1.0,NaN,3,15000,Hyderabad
4560,1,1.0,NaN,3,15000,Hyderabad


Any current_floor > total_floors left: 0
posted_date NaT: 0 | range: 2022-04-13 -> 2022-07-11

Missing values after rules:
 total_floors    4


## 3. Learned cleaning — `COPY VERBATIM INTO PHASE 5`

These compute a parameter from data, so they are **fit on X_train only** (in Phase 5) and live in the
`Pipeline`/`ColumnTransformer` (or, for row filtering, are computed from train and applied to train only).

* **`LogIQRClipper`** — learns Tukey fences on `log1p(x)` from train; `transform` clips values to them.
  Never drops rows, so it is safe to apply to val/test (with the *train* fences). Used for `Size` (lower side).
* **`train_only_far_out_mask`** — Tukey *far-out* fence (k=3) on `log(Rent/Size)` computed on train,
  returns a keep-mask **for train rows only**. Val/test are never filtered.
* **`SimpleImputer`** — median for numerics, most-frequent for categoricals (strategy chosen here, fit in Phase 5).

In [6]:
# ======================= PHASE-2 LEARNED CLEANING — COPY VERBATIM INTO PHASE 5 =======================
class LogIQRClipper(BaseEstimator, TransformerMixin):
    """Clip numeric columns to Tukey fences learned on log1p(x) of the TRAIN data.

    side: "lower", "upper" or "both". k: IQR multiplier. Fences are stored in fit; transform never
    drops rows, it only clips, so val/test can be transformed with the train-learned fences.
    """
    def __init__(self, k=1.5, side="lower"):
        self.k = k
        self.side = side

    def fit(self, X, y=None):
        L = np.log1p(np.asarray(X, dtype="float64"))
        q1, q3 = np.nanpercentile(L, 25, axis=0), np.nanpercentile(L, 75, axis=0)
        iqr = q3 - q1
        self.lower_ = np.expm1(q1 - self.k * iqr) if self.side in ("lower", "both") else np.full(L.shape[1], -np.inf)
        self.upper_ = np.expm1(q3 + self.k * iqr) if self.side in ("upper", "both") else np.full(L.shape[1], np.inf)
        self.n_features_in_ = L.shape[1]
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        A = np.asarray(X, dtype="float64")
        return np.clip(A, self.lower_, self.upper_)

    def get_feature_names_out(self, input_features=None):
        if input_features is not None:
            return np.asarray(input_features, dtype=object)
        return getattr(self, "feature_names_in_", np.asarray([f"x{i}" for i in range(self.n_features_in_)], dtype=object))


def train_only_far_out_mask(X_train, y_train, k=3.0):
    """Keep-mask for TRAIN rows: drop rows whose log(Rent/Size) lies beyond Tukey far-out fences (k=3)
    computed on the train rows themselves. Never call this on val/test."""
    rps = np.log(np.asarray(y_train, dtype="float64") / X_train["Size"].to_numpy(dtype="float64"))
    q1, q3 = np.percentile(rps, [25, 75]); iqr = q3 - q1
    lo, hi = q1 - k * iqr, q3 + k * iqr
    keep = (rps >= lo) & (rps <= hi)
    return keep, {"k": k, "rent_per_sqft_lower": float(np.exp(lo)), "rent_per_sqft_upper": float(np.exp(hi)),
                  "n_dropped": int((~keep).sum())}


NUM_IMPUTER = SimpleImputer(strategy="median")          # fit on X_train in Phase 5
CAT_IMPUTER = SimpleImputer(strategy="most_frequent")   # safety net (0 categorical NaNs today)
# =====================================================================================================

### 3a. Unit tests on synthetic data — proves the transformers learn from *fit* data only
(No real-data fitting here; that would pre-empt the Phase-5 split.)

In [7]:
rng = np.random.default_rng(RANDOM_STATE)
toy_train = pd.DataFrame({"Size": np.r_[rng.integers(400, 2000, 200), [10, 20]]})
toy_test = pd.DataFrame({"Size": [5, 15, 900, 50_000]})

clip = LogIQRClipper(k=1.5, side="lower").fit(toy_train)
print("Lower fence learned from toy_train:", clip.lower_.round(1), "| upper:", clip.upper_)
print("toy_test before:", toy_test["Size"].tolist(), "-> after:", clip.transform(toy_test).ravel().round(1).tolist())
assert np.all(clip.transform(toy_test) >= clip.lower_)
assert clip.transform(toy_test)[-1, 0] == 50_000, "upper side must be untouched when side='lower'"
assert len(clip.transform(toy_test)) == len(toy_test), "clipper must never drop rows"
refit = clone(clip).fit(toy_test)
assert not np.allclose(refit.lower_, clip.lower_), "fences depend only on the data passed to fit"

sz = rng.integers(400, 2000, 300)
toy_X = pd.DataFrame({"Size": np.r_[sz, [2500, 10]]})
toy_y = np.r_[sz * rng.uniform(10, 40, 300), [3_500_000, 15000]]     # last two mimic the real data errors
keep, info = train_only_far_out_mask(toy_X, toy_y)
print("Far-out mask on toy train: dropped positions", np.where(~keep)[0].tolist(), info)
assert (~keep).sum() == 2 and not keep[-1] and not keep[-2], "both planted errors must be flagged, nothing else"
print("All unit tests passed.")

Lower fence learned from toy_train: [321.7] | upper: [inf]
toy_test before: [5, 15, 900, 50000] -> after: [321.7, 321.7, 900.0, 50000.0]
Far-out mask on toy train: dropped positions [300, 301] {'k': 3.0, 'rent_per_sqft_lower': 2.357689766679826, 'rent_per_sqft_upper': 240.08833717621772, 'n_dropped': 2}
All unit tests passed.


## 4. Evidence behind the outlier decisions (descriptive, full data — look-only, like Phase 1)
The numbers below **justify the choice of method**; the actual fences are recomputed on train in Phase 5.

In [8]:
c = clean
log_size = np.log1p(c["Size"])
q1, q3 = log_size.quantile([.25, .75]); lo_sz = q1 - 1.5 * (q3 - q1); hi_sz = q3 + 1.5 * (q3 - q1)
low_sz = log_size < lo_sz
print(f"Size: log-IQR lower fence ≈ {np.expm1(lo_sz):.0f} sqft → {int(low_sz.sum())} rows below; "
      f"upper fence ≈ {np.expm1(hi_sz):.0f} sqft → {int((log_size > hi_sz).sum())} rows above")
print("\nIf tiny sizes were real flats, their rent should be far lower. Median rent by BHK:")
display(pd.DataFrame({"tiny-size rows": c[low_sz].groupby("BHK")["Rent"].median(),
                      "n_tiny": c[low_sz].groupby("BHK").size(),
                      "all other rows": c[~low_sz].groupby("BHK")["Rent"].median()}))
print("Tiny-size rows by City:", c[low_sz]["City"].value_counts().to_dict())
print("Large-size rows (> upper fence) median rent:", c[log_size > hi_sz]["Rent"].median(),
      "vs overall", c["Rent"].median(), "→ big homes have big rents, i.e. plausible")

Size: log-IQR lower fence ≈ 170 sqft → 186 rows below; upper fence ≈ 3864 sqft → 25 rows above

If tiny sizes were real flats, their rent should be far lower. Median rent by BHK:


,tiny-size rows,n_tiny,all other rows
BHK,,,
1,6000.0,63.0,9000.0
2,13000.0,93.0,15000.0
3,18750.0,28.0,32000.0
4,12500.0,2.0,140000.0
5,NaN,NaN,310000.0
6,NaN,NaN,47500.0


Tiny-size rows by City: {'Delhi': 118, 'Hyderabad': 41, 'Bangalore': 11, 'Kolkata': 9, 'Chennai': 6, 'Mumbai': 1}
Large-size rows (> upper fence) median rent: 190000.0 vs overall 16000.0 → big homes have big rents, i.e. plausible


In [9]:
log_rent = np.log1p(c["Rent"])
q1, q3 = log_rent.quantile([.25, .75]); hi_r = q3 + 1.5 * (q3 - q1)
high = log_rent > hi_r
print(f"log1p(Rent) above 1.5·IQR fence: {int(high.sum())} rows")
print("  by City            :", c[high]["City"].value_counts().to_dict())
print("  by Point of Contact:", c[high]["Point of Contact"].value_counts().to_dict())
print("  by Furnishing      :", c[high]["Furnishing Status"].value_counts().to_dict())
print("  → the high tail is explained by features (Mumbai, agent-listed) → genuine market segment, keep.")

rps = np.log(c["Rent"] / c["Size"]); q1, q3 = rps.quantile([.25, .75]); i = q3 - q1
for k in (1.5, 3.0):
    n_out = int(((rps < q1 - k * i) | (rps > q3 + k * i)).sum())
    print(f"\nlog(Rent/Size) beyond k={k} fences: {n_out} rows")
print("Rows beyond the k=3 far-out fence (look only — they are removed by the train-only rule IF they land in train):")
display(c[(rps < q1 - 3 * i) | (rps > q3 + 3 * i)][["Rent", "Size", "BHK", "Bathroom", "City", "Area Locality"]]
        .assign(rent_per_sqft=lambda d: (d["Rent"] / d["Size"]).round(1)))

log1p(Rent) above 1.5·IQR fence: 114 rows
  by City            : {'Mumbai': 89, 'Delhi': 9, 'Chennai': 7, 'Hyderabad': 5, 'Bangalore': 4}
  by Point of Contact: {'Contact Agent': 104, 'Contact Owner': 10}
  by Furnishing      : {'Semi-Furnished': 58, 'Furnished': 40, 'Unfurnished': 16}
  → the high tail is explained by features (Mumbai, agent-listed) → genuine market segment, keep.

log(Rent/Size) beyond k=1.5 fences: 54 rows

log(Rent/Size) beyond k=3.0 fences: 2 rows
Rows beyond the k=3 far-out fence (look only — they are removed by the train-only rule IF they land in train):


,Rent,Size,BHK,Bathroom,City,Area Locality,rent_per_sqft
1833,3500000,2500,3,3,Bangalore,marathahalli,1400.0
4644,15000,10,3,3,Hyderabad,ghatkesar nh 2 2,1500.0


## 5. Column removals

In [10]:
summary = pd.DataFrame({"n_unique": raw.nunique(), "unique_ratio": (raw.nunique() / len(raw)).round(3)})
print("ID / index-like columns (unique_ratio > 0.95):", summary.index[summary.unique_ratio > 0.95].tolist())
print("Constant columns:", summary.index[summary.n_unique == 1].tolist())
print("Leakage columns (from Phase-1 D12): []")
print("→ No column is removed for being an ID, constant or leaky.")
print("  Raw 'Floor' / 'Posted On' strings are superseded by parsed columns; their removal is finalised in Phase 4.")
print("\nFinal check — raw frame untouched:", raw.shape, "| nothing written to disk.")

ID / index-like columns (unique_ratio > 0.95): []
Constant columns: []
Leakage columns (from Phase-1 D12): []
→ No column is removed for being an ID, constant or leaky.
  Raw 'Floor' / 'Posted On' strings are superseded by parsed columns; their removal is finalised in Phase 4.

Final check — raw frame untouched: (4746, 12) | nothing written to disk.


## 6. Phase-2 decisions

### Stateless rules (pre-split, `apply_stateless_rules`) — 4,746 → **4,737 rows**
| Rule | Rows affected | Decision |
|---|---|---|
| R1 `strip_text` | 4 localities trimmed; locality text normalised (2,235 → 2,228 unique) | apply |
| R2 `validity` (Rent>0, Size>0, BHK≥1, Bathroom≥1) | 0 removed (guard rule) | apply |
| R3 `dedup_relisted` (all cols except `Posted On`) | **9 removed** (8 seen in Phase 1 + 1 exposed by R1: "GH Society" vs "gh society") | apply — prevents the same listing in train and test |
| R4 `parse_floor` | 4 rows → `total_floors`=NaN; 2 rows ("8 out of 5", "2 out of 1") → total := current | apply |
| R5 `parse_date` | 0 NaT | apply |

### Learned cleaning (defined here, **fit on X_train only** in Phase 5)
| Item | Decision |
|---|---|
| Imputation | `SimpleImputer(median)` for numerics (only `total_floors`, 4 NaN); `most_frequent` for categoricals as a safety net. Fit on train. |
| **Size errors** (≈186 rows below the log-IQR lower fence, ≈170 sqft; 118 of them in Delhi) | **Keep rows, clip the value**: `LogIQRClipper(k=1.5, side="lower")` on `Size`, fences learned on train, applied to all splits. Evidence: tiny-size rows have *normal* rents for their BHK, so the **Size field** is wrong, not the listing. Upper side left alone — big homes have big rents. |
| **Rent tail** (114 rows above the log-IQR fence) | **Keep.** 89 are Mumbai and 104 are agent-listed — a real segment explained by the features. Handled by `log1p` (Phase 3) and robust models, not by deletion. |
| **Far-out data errors** (2 rows beyond the k=3 fence on log rent-per-sqft, including **₹35,00,000 for 2,500 sqft**, ≈₹1,400/sqft) | `train_only_far_out_mask(k=3)`: fences computed on **train**, rows dropped from **train only**. Val/test are never filtered. If such a row falls in val/test it stays, and its effect on rupee RMSE is reported honestly. |
| Rare categories (D9) | Unchanged: `OneHotEncoder(min_frequency=…, handle_unknown="infrequent_if_exist")` fit on train in Phase 5. |

### Columns removed
None as ID/constant/leakage (none exist). Raw `Floor` and `Posted On` strings are superseded by parsed columns; dropping them is finalised in Phase 4.

### Effects on later phases
* **Phase 3**: run the target analysis on the rule-cleaned 4,737 rows.
* **Phase 4**: any feature built from `Size` (e.g. `size_per_bhk`) must be computed **after** the clipper, i.e. inside the fitted pipeline, or it inherits the bad sizes.
* **Phase 5**: order is fixed: rules → split → `train_only_far_out_mask` on train → fit the ColumnTransformer (clipper, imputers, encoders) on the filtered train → transform all splits. The fingerprint is taken after this filtering.